In [0]:
catalog = "projeto_cinedata"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

from pyspark.sql import functions as F
from pyspark.sql.window import Window

win_sk = Window.orderBy(F.lit(1))

=============================================================================================Modelagem Dimensional (Star Schema)

============================================================================================= gold.dim_movies

In [0]:
df_silver_info = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_silver_info
    .select("id_filme", "titulo", "data_lancamento", "ano_lancamento", 
            "duracao_minutos", "idioma_original", "status_filme", "sinopse")
    .dropDuplicates(["id_filme"])
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .withColumn("sk_movie_id", F.monotonically_increasing_id().cast("bigint"))
)
df_dim_movies = df_dim_movies.select("sk_movie_id", *[c for c in df_dim_movies.columns if c != "sk_movie_id"])

============================================================================================= gold.dim_genres

In [0]:
df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

window_genres = Window.partitionBy(F.lit(1)).orderBy("nome_genero")

df_dim_genres = (
    df_silver_generos
    .select(F.col("genero").alias("nome_genero"))
    .dropDuplicates()
    .withColumn("sk_genre_id", F.row_number().over(window_genres).cast("bigint"))
).select("sk_genre_id", "nome_genero")

============================================================================================= gold.dim_companies

In [0]:
df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"), 
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
    .withColumn("sk_person_id", F.monotonically_increasing_id().cast("bigint"))
).select("sk_person_id", "nome_pessoa", "tipo_pessoa")

df_dim_companies = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", F.monotonically_increasing_id().cast("bigint"))
).select("sk_company_id", "nome_produtora")

for table_name, df in [("dim_movies", df_dim_movies), ("dim_genres", df_dim_genres), 
                       ("dim_people", df_dim_people), ("dim_companies", df_dim_companies)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.{table_name}")
    print(f"[OK] {table_name} criada com sucesso!")

============================================================================================= Tabelas-Ponte (Bridges)

In [0]:
dim_movies = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")
dim_genres = spark.table(f"{gold_schema}.dim_genres")
dim_people = spark.table(f"{gold_schema}.dim_people")
dim_companies = spark.table(f"{gold_schema}.dim_companies")

df_bridge_genre = (
    df_silver_generos
    .join(dim_movies, "id_filme", "inner")
    .join(dim_genres, df_silver_generos.genero == dim_genres.nome_genero, "inner")
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)

df_bridge_person = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dim_movies, "id_filme", "inner")
    .join(dim_people, 
          (df_silver_pessoas.nome_entidade == dim_people.nome_pessoa) & 
          (df_silver_pessoas.tipo_entidade == dim_people.tipo_pessoa), "inner")
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
)

df_bridge_company = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(dim_movies, "id_filme", "inner")
    .join(dim_companies, df_silver_pessoas.nome_entidade == dim_companies.nome_produtora, "inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)

for table_name, df in [("bridge_movie_genre", df_bridge_genre), 
                       ("bridge_movie_person", df_bridge_person), 
                       ("bridge_movie_company", df_bridge_company)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.{table_name}")
    print(f"[OK] {table_name} criada com sucesso!")

============================================================================================= gold.dim_reviews

In [0]:
df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

df_dim_reviews = (
    df_silver_reviews
    .groupBy("id_filme")
    .agg(
        F.count("id_filme").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
    .join(dim_movies, "id_filme", "inner")
    .withColumn("sk_review_id", F.monotonically_increasing_id().cast("bigint"))
).select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")

============================================================================================= gold.fact_movies_perfomance

In [0]:
df_silver_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_eng = spark.table(f"{silver_schema}.tb_metricas_engajamento")

df_fact_performance = (
    dim_movies
    .join(df_silver_fin, "id_filme", "left")
    .join(df_silver_eng, "id_filme", "left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int")
    )
)

for table_name, df in [("dim_reviews", df_dim_reviews), ("fact_movies_performance", df_fact_performance)]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.{table_name}")
    print(f"[OK] {table_name} criada com sucesso!")

display(df_fact_performance.limit(10))

============================================================================================= Tabela gold_genai_movies_context

1. Agregação de Atores e Diretores via Tabelas-Ponte

In [0]:
dim_movies = spark.table(f"{gold_schema}.dim_movies")
fact_perf = spark.table(f"{gold_schema}.fact_movies_performance")
dim_people = spark.table(f"{gold_schema}.dim_people")
bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")

df_pessoas_agrupadas = (
    bridge_person
    .join(dim_people, "sk_person_id", "inner")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.collect_list(F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa")))).alias("atores_principais"),
        F.concat_ws(", ", F.collect_list(F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa")))).alias("diretor")
    )
)

2. Cruzamento Geral (Join)

In [0]:
df_contexto_bruto = (
    dim_movies
    .join(fact_perf, "sk_movie_id", "left")
    .join(df_pessoas_agrupadas, "sk_movie_id", "left")
)

3. Tratamento de Nulos (Fallback) e Concatenação Inteligente

In [0]:
df_gold_genai = (
    df_contexto_bruto
    .withColumn("safe_titulo", F.coalesce("titulo", F.lit("Título Desconhecido")))
    .withColumn("safe_ano", F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano não informado")))
    .withColumn(
        "safe_receita", 
        F.when(F.col("receita_usd").isNotNull(), F.concat(F.lit("US$ "), F.col("receita_usd").cast("string"))).otherwise(F.lit("um valor não divulgado"))
    )
    .withColumn(
        "safe_orcamento", 
        F.when(F.col("orcamento_usd").isNotNull(), F.concat(F.lit("US$ "), F.col("orcamento_usd").cast("string"))).otherwise(F.lit("um orçamento não divulgado"))
    )
    .withColumn(
        "safe_atores", 
        F.when((F.col("atores_principais").isNotNull()) & (F.col("atores_principais") != ""), F.col("atores_principais")).otherwise(F.lit("elenco não informado"))
    )
    .withColumn(
        "safe_diretor", 
        F.when((F.col("diretor").isNotNull()) & (F.col("diretor") != ""), F.col("diretor")).otherwise(F.lit("diretor não informado"))
    )
    .withColumn("safe_sinopse", F.coalesce("sinopse", F.lit("Sinopse indisponível.")))
    
    .withColumn(
        "llm_context_document",
        F.concat(
            F.lit("O filme "), F.col("safe_titulo"),
            F.lit(", lançado no ano de "), F.col("safe_ano"),
            F.lit(", faturou "), F.col("safe_receita"),
            F.lit(" e teve um custo de "), F.col("safe_orcamento"),
            F.lit(". Estrelado por "), F.col("safe_atores"),
            F.lit(" e dirigido por "), F.col("safe_diretor"),
            F.lit(", o filme possui a seguinte sinopse: "), F.col("safe_sinopse")
        )
    )
    .select(
        F.col("id_filme").alias("movie_id"),
        F.col("titulo").alias("title"),
        "llm_context_document"
    )
)

4. Persistência da Tabela Gold

In [0]:
(
    df_gold_genai.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{gold_schema}.gold_genai_movies_context")
)

print("[OK] Tabela gold_genai_movies_context gerada com sucesso!")
display(df_gold_genai.limit(10))

=============================================================================================

In [0]:
spark.sql("USE CATALOG projeto_cinedata")

1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
display(spark.sql("""
    SELECT 
        CAST(SUM(receita_brl) AS DECIMAL(25,2)) AS receita_total_brl 
    FROM gold.fact_movies_performance
"""))

2. Quais são os 5 filmes com maior popularidade?

In [0]:
display(spark.sql("""
    SELECT 
        m.titulo, 
        f.popularidade 
    FROM gold.fact_movies_performance f
    INNER JOIN gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    ORDER BY f.popularidade DESC NULLS LAST
    LIMIT 5
"""))

3. Quantos filmes cada gênero possui? Liste do maior para o menor.

In [0]:
display(spark.sql("""
    SELECT 
        g.nome_genero, 
        COUNT(b.sk_movie_id) AS qtd_filmes 
    FROM gold.bridge_movie_genre b
    INNER JOIN gold.dim_genres g ON b.sk_genre_id = g.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

4. Ranking dos 10 filmes de maior receita (US$ e R$).

In [0]:
display(spark.sql("""
    SELECT 
        m.titulo, 
        f.receita_usd, 
        f.receita_brl,
        RANK() OVER(ORDER BY f.receita_usd DESC) AS rank_receita
    FROM gold.fact_movies_performance f
    INNER JOIN gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
    ORDER BY rank_receita
    LIMIT 10
"""))

5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

In [0]:
display(spark.sql("""
    WITH DataLimite AS (
        -- Descobre a data de lançamento mais recente no banco (ignorando o futuro)
        SELECT MAX(data_lancamento) AS max_date 
        FROM gold.dim_movies 
        WHERE data_lancamento <= CURRENT_DATE()
    )
    SELECT 
        p.nome_pessoa AS ator, 
        COUNT(b.sk_movie_id) AS participacoes 
    FROM gold.bridge_movie_person b
    INNER JOIN gold.dim_people p ON b.sk_person_id = p.sk_person_id
    INNER JOIN gold.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN DataLimite dl
    WHERE p.tipo_pessoa = 'Ator' 
      AND m.data_lancamento >= ADD_MONTHS(dl.max_date, -24)
      AND m.data_lancamento <= dl.max_date
    GROUP BY p.nome_pessoa
    ORDER BY participacoes DESC
    LIMIT 1
"""))

6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?

In [0]:
display(spark.sql("""
    WITH DataLimite AS (
        SELECT MAX(data_lancamento) AS max_date 
        FROM gold.dim_movies 
        WHERE data_lancamento <= CURRENT_DATE()
    )
    SELECT 
        c.nome_produtora, 
        CAST(SUM(f.lucro_usd) AS DECIMAL(25,2)) AS lucro_total_usd 
    FROM gold.bridge_movie_company b
    INNER JOIN gold.dim_companies c ON b.sk_company_id = c.sk_company_id
    INNER JOIN gold.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    INNER JOIN gold.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    CROSS JOIN DataLimite dl
    WHERE m.data_lancamento >= ADD_MONTHS(dl.max_date, -60)
      AND m.data_lancamento <= dl.max_date
    GROUP BY c.nome_produtora
    ORDER BY lucro_total_usd DESC NULLS LAST
    LIMIT 1
"""))